# Dvouvrstvý perceptron s využitím autogradu

- Úkolem cvičení je natrénovat opět **dvouvrstvý perceptron** pro klasifikaci obrázků na datasetu CIFAR-10, **tentokrát ovšem za použití modulu `ans.autograd` z minulého cvičení**.
- Vytvoříme novou třídu `ans.classification.TwoLayerPerceptronAutograd`, která bude obsahovat pouze dopředný průchod. Zpětný průchod zajistí automaticky implementace `ans.autograd.Variable.backprop` implementovaná v minulém cvičení.
- Vytvoříme rovněž nový modul `ans.nn`, do kterého budeme postupně přidávat funkcionalitu související s neuronovými sítěmi, abychom nemuseli neustále opakovat shodné bloky kódu.

**Poznámka**
- Tento notebook nezapíná [autoreload extension](https://ipython.org/ipython-doc/3/config/extensions/autoreload.html), protože testování využívá type checking pomocí `isinstance` a [to při reloadu modulu selhává](https://github.com/ipython/ipython/issues/12399).
- Při každé modifikaci modulu `ans.autograd` je proto bohužel nutné notebook restartovat a spustit celý kód znovu (např. "Run All") 😟

In [1]:
import sys

sys.path.append("..")  # import tests

import numpy as np
import PIL
import torch
import torchvision

import ans
from tests import test_perceptron_autograd

print("test")

test


# Příprava diferencovatelných operací

- Z minulého cvičení ma `Variable` implementované pouze operace `+`, `-`, `*` a `/`.
- Pro vícevrstvý perceptron (viz cvičení [two_layer_perceptron](two_layer_perceptron.ipynb)) optimalizovaný softmax křížovou entropií budeme ještě potřebovat alespoň
  - maticové násobení `@` pro vnitřní vrstvy perceptronu,
  - nelinearitu `sigmoid`,
  - `log`, `sum`, `exp` a `__getitem__` pro výpočet lossu.

## Maticové násobení `z = x @ y`

- Budeme uvažovat pouze minimálně dvourozměrné tensory, tj. matice x matice a výše.

**Dopředný průchod**

$$\boldsymbol{z} = \boldsymbol{x} \cdot \boldsymbol{y}$$

- $\boldsymbol{x} = [x_{\ldots,n,d}]$ je nejméně dvourozměrný tensor s rozměry $\ldots \times N \times D$,
- $\boldsymbol{y} = [y_{\ldots,d,k}]$ je nejméně dvourozměrný tensor s rozměry $\ldots \times D \times K$,
- $\boldsymbol{z} = [z_{\ldots,n,k}]$ je nejméně dvourozměrný tensor s rozměry $\ldots \times N \times K$,

**Zpětný průchod**

$$
\begin{align*}
    \overline{\boldsymbol{x}} &= \overline{\boldsymbol{z}} \cdot \boldsymbol{y}^\top \\
    \overline{\boldsymbol{y}} &= \boldsymbol{x}^\top \cdot \overline{\boldsymbol{z}}
\end{align*}
$$

- transpozicí $\boldsymbol{x}^\top$ pro vícerozměrné tensory se myslí prohození posledních dvou dimenzí $\ldots \times N \times D \rightarrow \ldots \times D \times N$

**Bonus**

- Zobecněte tak aby operátor `x @ y` fungoval i pro vektor (1D tensor) x matice a všechny ostatní kombinace podporované v PyTorchi.

### TODO: implementujte operátory [`ans.autograd.Variable.__matmul__`](../ans/autograd.py) a [`ans.autograd.Variable.__rmatmul__`](../ans/autograd.py)

In [2]:
test_perceptron_autograd.TestMatMul.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.061s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [3]:
test_perceptron_autograd.TestMatMulGeneral.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.074s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Sigmoid `z = x.sigmoid()`

**Dopředný průchod**

$$z = \frac{1}{1 + e^{-x}}$$
- $x$ je reálné číslo (skalár)
- $z$ je reálně číslo (skalár)

**Zpětný průchod**

$$\overline{x} = \overline{z} \cdot z \cdot (1 - z)$$

### TODO: implementujte metodu [`ans.autograd.Variable.sigmoid`](../ans/autograd.py)

In [4]:
test_perceptron_autograd.TestSigmoid.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.020s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Umocňování `z = x.exp()`

**Dopředný průchod**

$$
z = e^{x}
$$
- $x$ je reálné číslo (skalár)
- $z$ je reálně číslo (skalár)

**Zpětný průchod**

$$
\overline{x} = \overline{z} \cdot z
$$

### TODO: implementujte metodu [`ans.autograd.Variable.exp`](../ans/autograd.py)

In [5]:
test_perceptron_autograd.TestExp.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.011s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Přirozený logaritmus `z = x.log()`

- Nutné pro výpočet křížové entropie, kde se loss počítá jako logaritmus z pravděpodobnosti predikované pro target třídu.

**Dopředný průchod**

$$
z = \ln x
$$
- $x$ je reálné číslo (skalár)
- $z$ je reálně číslo (skalár)

**Zpětný průchod**

$$
\overline{x} = \overline{z} \cdot \frac{1}{x}
$$

### TODO: implementujte metodu [`ans.autograd.Variable.log`](../ans/autograd.py)

In [6]:
test_perceptron_autograd.TestLog.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.011s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Výběr prvku `z = x[indices]`

- Nutné pro výpočet křížové entropie, kde se loss počítá jako $l = \ln \widehat{p}_y$, kde $\widehat{p}_y$ je pravděpodobnost predikovaná modelem pro target třídu $y$.

**Dopředný průchod**

- Operace je implementovaná v PyTorch jako `z = x[ids]`, kde `ids` může být `int`, `tuple[int,...]`, `list[int,...]` nebo `torch.Tensor`.
- Operaci si zjednodušíme a výběr omezíme na podmnožinu prvků `x`, kde se navíc nemůže opakovat výběr stejného prvku.
- Příklad:
  ``` python
  x = torch.tensor([0.5, 0.6, 0.7])
  z = x[[0]]  # z = torch.tensor([0.5]) ... ok
  z = x[[0, 1]]  # z = torch.tensor([0.5, 0.6]) ... ok
  z = x[[0, 0]]  # z = torch.tensor([0.5, 0.5]) ... validni v PyTorch, ale zde pro jednoduchost neuvazujeme (opakuje se prvek 0)
  z = x[[False, True, True]]  # z = torch.tensor([0.6, 0.7]) ... ok
  z = x[:2]  # z = torch.tensor([0.5, 0.6]) ... ok
  ...
  ```

**Zpětný průchod**

- Lokální gradient na *vybrané* prvky je jedna.
- Lokální gradient na *nevybrané* prvky je nula.

In [7]:
test_perceptron_autograd.TestGetItem.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.050s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Součet prvků `z = x.sum(dim=..., keepdim=...)`

- Nutné pro výpočet softmaxu, kdy potřebujeme součet přes pravděpodobnosti tříd.

**Dopředný průchod**

- Operace je implementovaná v PyTorch jako `z = x.sum(...)`
- Podporované musí být oba parametry `dim` a `keepdim`, viz <https://pytorch.org/docs/stable/generated/torch.sum.html>

**Zpětný průchod**

- Lokální gradient na všechny prvky je jedna.

### TODO: implementujte metodu [`ans.autograd.Variable.sum`](../ans/autograd.py)

In [8]:
test_perceptron_autograd.TestSum.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.031s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## (Bonus) Průměr prvků `z = x.mean(dim=..., keepdim=...)`

- Podobné jako `x.sum(...)`, pouze s průměrem namísto součtu.

### TODO: implementujte metodu [`ans.autograd.Variable.mean`](../ans/autograd.py)

In [9]:
test_perceptron_autograd.TestMean.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ERROR

ERROR: test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\tests\test_perceptron_autograd.py", line 69, in test_operation
    z_var = self.forward(x_var, **kwargs)  # with Variables
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\tests\test_perceptron_autograd.py", line 158, in forward
    return x.mean(**kwargs)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Diabl\ANS\ans-2026\assignments\..\ans\autograd.py", line 294, in mean
    raise NotImplementedError
NotImplementedError

----------------------------------------------------------------------
Ran 1 test in 0.005s

FAILED (errors=1)



*** TEST PARAMS CONFIGURATION THAT FAILED ***
dtype=torch.float32, device=cpu, shape=(2,), kwargs={}


<unittest.runner.TextTestResult run=1 errors=1 failures=0>

# Třída `TwoLayerPerceptronAutograd`

- Cílem cvičení je natrénovat model dvouvrstvého perceptronu bez explicitního kódování zpětného průchodu, tj. s využitím autogradu.
- Model bude imeplementovat třída `ans.classification.TwoLayerPerceptronAutograd`.
- Parametry klasifikátoru budou reprezentovány jako `ans.autograd.Variable`, viz následující tabulku  
  | atribut   | typ                     | značení                                                            | rozměr       |
  |-----------|-------------------------|--------------------------------------------------------------------|--------------|
  | `weight1` | `ans.autograd.Variable` | $\boldsymbol{w}^{(1)} = \left[w_{d,h}^{(1)}\right]$                | $D \times H$ |
  | `bias1`   | `ans.autograd.Variable` | $\boldsymbol{b}^{(1)} = \left[b_1^{(1)}, \ldots, b_H^{(1)}\right]$ | $H$          |
  | `weight2` | `ans.autograd.Variable` | $\boldsymbol{w}^{(2)} = \left[w_{d,k}^{(2)}\right]$                | $H \times K$ |
  | `bias2`   | `ans.autograd.Variable` | $\boldsymbol{b}^{(2)} = \left[b_1^{(2)}, \ldots, b_K^{(2)}\right]$ | $K$          |
- Třída bude obsahovat pouze kód pro dopředný průchod (průchod sítí a výpočet lossu).
- Zpětný průchod bude volán jako
  ``` python
  # vynulovani gradientu
  ...
  # zpetna propagace
  loss.backprop()
  ```
- Po jeho provedení budou vyplněny atributy `.grad` všechny parametrů modelu, což umožní update pomocí SGD.
- **Důležité je před každým zpětným průchodem vynulovat gradienty parametrů, aby nedocházelo k jejich akumulaci přes různé dávky (minibatche)**, viz např. <https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch>.

**Funkce `softmax_cross_entropy`**
- Výpočet lossu implementujte jako funkci `ans.classification.softmax_cross_entropy`, aby se nemusely opakovat stejné kusy kódu v `TwoLayerPerceptronAutograd.train_step` a `TwoLayerPerceptronAutograd.val_Step`.
- Funkce by měla být agnostická vůči typu `logits` parametru:
  - pokud je `torch.Tensor`, výsledný loss bude rovněž `torch.Tensor`,
  - pokud je `Variable`, výsledný loss bude rovněž `Variable`.

### TODO: implementuje funkci [`ans.classification.softmax_cross_entropy`](../ans/classification.py).

In [10]:
test_perceptron_autograd.TestSoftmaxCrossEntropy.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_sce (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_sce) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.017s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje metodu [`ans.classification.TwoLayerPerceptronAutograd.__init__`](../ans/classification.py).

In [11]:
test_perceptron_autograd.TestInit.eval()

test_init_params (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_init_params) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.010s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptronAutograd.train_step`](../ans/classification.py).

In [12]:
test_perceptron_autograd.TestTrainStep.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.016s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptronAutograd.val_step`](../ans/classification.py).

In [13]:
test_perceptron_autograd.TestValStep.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.015s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Preprocessing dat

- Pro trénování bude potřeba opět ještě preprocessing dat, podobně jako v minulých cvičeních.
- Použijeme shodný preprocessing jako ve cvičeních [linear_classification](linear_classification.ipynb) a [two_two_layer_perceptron](two_layer_perceptron.ipynb), tj. zploštení obrázků do vektorů a převod hodnot pixelů 0..255 do rozsahu 0..1 prostým vydělením 255.

### TODO: implementuje funkci `preprocess`.

In [14]:
def preprocess(img: PIL.Image) -> torch.Tensor:
    """
    Args:
        img: image of type PIL.Image
    Returns:
        x: 1-dimensional tensor; shape (num_pixels * num_channels,), dtype float32
    """

    ########################################
    # TODO: implement

    shape = np.array(img)
    x = torch.tensor(shape, dtype = torch.float32) / 255.0
    x = x.reshape(-1)
    
    return x

In [15]:
test_perceptron_autograd.TestPreprocess.eval(preprocess_fn=preprocess)

test_preprocess (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_preprocess) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# Trénování klasifikátoru

- Pokud inicializujete parametry ve stejném pořadí a shodným způsobem jako ve cvičení [two_two_layer_perceptron](two_layer_perceptron.ipynb) (a pouze "obalíte" do `Variable`), měli byste dostat naprosto stejná čísla včetně progressu lossu!
- Nejlepší model uložte jako
  ``` python
  model.save('../output/two_layer_perceptron_autograd_weights.pt')
  ```

### TODO: Natrénujte dvouvrstvý perceptron tak, aby dosáhl alespoň 45 % (bonusově 50 %) *validační* accuracy.

In [19]:
ans.utils.seed_everything(0)

########################################
batch_size = 64
hidden_size = 128
learning_rate = 0.05
num_epochs = 40

device = torch.device("cpu")

train_dataset = torchvision.datasets.CIFAR10(root="../data", train=True, transform=preprocess)
val_dataset = torchvision.datasets.CIFAR10(root="../data", train=False, transform=preprocess)

sample_img, _ = train_dataset[0]
in_size = sample_img.numel()
out_size = len(train_dataset.classes)

train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True, device=device)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False, device=device)

model = ans.classification.TwoLayerPerceptronAutograd(
    in_size=in_size,
    hidden_size=hidden_size,
    out_size=out_size,
    weight_scale=1e-2,
)

os.makedirs("../output", exist_ok=True)
best_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = ans.classification.train_epoch(model, train_loader, learning_rate=learning_rate)
    val_loss, val_acc = ans.classification.validate(model, val_loader)

    print(
        f"epoch {epoch + 1}: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, "
        f"val_loss={val_loss:.5f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        model.save("../output/two_layer_perceptron_autograd_weights.pt")
#batch = 256, hidden = 256, learning = 0.05, epoch = 25 -> 0.402
#batch = 64, hidden = 128, learning = 0.05, epoch = 40 -> 0.505
print("Best val acc:", best_acc)
# ENDTODO
########################################

epoch 1: train_loss=2.13543, train_acc=0.208, val_loss=1.97823, val_acc=0.285
epoch 2: train_loss=1.92752, train_acc=0.308, val_loss=1.88910, val_acc=0.324
epoch 3: train_loss=1.85422, train_acc=0.341, val_loss=1.83697, val_acc=0.338
epoch 4: train_loss=1.80306, train_acc=0.360, val_loss=1.83383, val_acc=0.336
epoch 5: train_loss=1.76535, train_acc=0.374, val_loss=1.77629, val_acc=0.355
epoch 6: train_loss=1.73096, train_acc=0.387, val_loss=1.74462, val_acc=0.371
epoch 7: train_loss=1.70103, train_acc=0.397, val_loss=1.72051, val_acc=0.391
epoch 8: train_loss=1.67404, train_acc=0.409, val_loss=1.70948, val_acc=0.382
epoch 9: train_loss=1.64795, train_acc=0.419, val_loss=1.65347, val_acc=0.407
epoch 10: train_loss=1.62406, train_acc=0.429, val_loss=1.77394, val_acc=0.351
epoch 11: train_loss=1.60372, train_acc=0.436, val_loss=1.59734, val_acc=0.430
epoch 12: train_loss=1.58437, train_acc=0.440, val_loss=1.65961, val_acc=0.407
epoch 13: train_loss=1.56673, train_acc=0.447, val_loss=1.592

In [20]:
test_perceptron_autograd.TestValAccuracy45.eval(preprocess_fn=preprocess)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ok

----------------------------------------------------------------------
Ran 1 test in 7.838s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [21]:
test_perceptron_autograd.TestValAccuracy50.eval(preprocess_fn=preprocess)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ok

----------------------------------------------------------------------
Ran 1 test in 7.787s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>